# 📊 Исследовательский анализ данных о дефектах ПО

## Анализ метрик кода и их связи с дефектами

Этот ноутбук выполняет исследовательский анализ данных (EDA) для датасета предсказания дефектов в программном обеспечении. Мы исследуем метрики кода, их распределения и корреляции с наличием дефектов.

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import sys

# Добавляем путь к проекту
sys.path.insert(0, str(Path.cwd().parent))

# Импортируем наши модули
from etl.load import DataLoader

# Настройки отображения
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Параметры отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✅ Библиотеки загружены")

## 1. Загрузка данных

Загружаем финальные обработанные данные после ETL-конвейера.

In [ ]:
# Загружаем данные
loader = DataLoader()
data = loader.load_from_processed('clean_defect_data.parquet')

if data is None:
    # Если нет обработанных, пробуем финальные
    final_path = Path("data/final/final_defect_data.csv")
    if final_path.exists():
        data = pd.read_csv(final_path)
        print("Загружены финальные данные")
    else:
        print("❌ Данные не найдены. Сначала запустите ETL-конвейер.")
else:
    print("✅ Данные успешно загружены")

print(f"\nРазмер датасета: {data.shape}")
print(f"Количество признаков: {data.shape[1]}")
print(f"Количество образцов: {data.shape[0]}")

In [ ]:
# Первые несколько строк
data.head()

## 2. Общая информация о данных

Посмотрим на типы данных, пропуски и базовую статистику.

In [ ]:
# Информация о данных
print("📊 ИНФОРМАЦИЯ О ДАННЫХ")
print("=" * 50)
data.info()

In [ ]:
# Проверка пропущенных значений
missing = data.isnull().sum()
missing_pct = (missing / len(data)) * 100

missing_df = pd.DataFrame({
    'Пропущено': missing,
    'Процент': missing_pct
}).sort_values('Пропущено', ascending=False)

print("\n🔍 ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ")
print("=" * 50)
missing_df[missing_df['Пропущено'] > 0]

In [ ]:
# Базовая статистика
print("\n📈 СТАТИСТИКА ЧИСЛОВЫХ ПРИЗНАКОВ")
print("=" * 50)
data.describe()

## 3. Анализ целевой переменной

Исследуем распределение дефектов в данных.

In [ ]:
# Распределение целевой переменной
target_col = 'defect_label' if 'defect_label' in data.columns else 'Defect'

if target_col in data.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # График распределения
    ax1 = axes[0]
    class_counts = data[target_col].value_counts()
    colors = ['#2ecc71', '#e74c3c']
    
    wedges, texts, autotexts = ax1.pie(
        class_counts.values, 
        labels=['Без дефектов (0)', 'С дефектами (1)'],
        autopct='%1.1f%%',
        colors=colors,
        startangle=90,
        explode=(0.05, 0.05)
    )
    ax1.set_title('Распределение классов', fontsize=14, fontweight='bold')
    
    # Настройка текста
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(12)
        autotext.set_fontweight('bold')
    
    # Столбчатая диаграмма
    ax2 = axes[1]
    bars = ax2.bar(class_counts.index.astype(str), class_counts.values, color=colors)
    ax2.set_xlabel('Класс')
    ax2.set_ylabel('Количество')
    ax2.set_title('Количество образцов по классам', fontsize=14, fontweight='bold')
    
    # Добавляем значения на столбцы
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../images/class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Вывод статистики
    print("\n📊 Статистика классов:")
    for label, count in class_counts.items():
        pct = (count / len(data)) * 100
        print(f"  Класс {label}: {count} образцов ({pct:.2f}%)")
    
    # Проверка на дисбаланс
    minority_ratio = min(class_counts.values) / len(data)
    if minority_ratio < 0.2:
        print("\n⚠️  Обнаружен дисбаланс классов!")
        print(f"   Доля минорного класса: {minority_ratio:.2%}")
        print("   Рекомендуется использовать методы балансировки для ML.")
else:
    print("❌ Целевая переменная не найдена")

## 4. Анализ метрик кода

Исследуем распределения основных метрик программного кода.

In [ ]:
# Выбираем основные метрики для анализа
code_metrics = ['loc', 'cyclo_complexity', 'halstead_volume', 'halstead_difficulty', 
                'branch_count', 'condition_count']

# Проверяем, какие метрики есть в данных
available_metrics = [m for m in code_metrics if m in data.columns]

if available_metrics:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for i, metric in enumerate(available_metrics[:6]):  # Максимум 6 графиков
        ax = axes[i]
        
        # Разделяем по классам
        for class_label, color in zip([0, 1], ['#2ecc71', '#e74c3c']):
            class_data = data[data[target_col] == class_label][metric]
            
            # Логарифмическая шкала для скошенных распределений
            if metric in ['loc', 'halstead_volume']:
                class_data = np.log1p(class_data)
                xlabel = f'log({metric}+1)'
            else:
                xlabel = metric
            
            ax.hist(class_data, bins=30, alpha=0.6, color=color, 
                   label=f'Класс {class_label}', density=True)
        
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Плотность')
        ax.set_title(f'Распределение {metric}', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Убираем лишние подграфики
    for j in range(len(available_metrics), 6):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.savefig('../images/metrics_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("❌ Метрики кода не найдены")

## 5. Корреляционный анализ

Исследуем взаимосвязи между метриками кода и целевой переменной.

In [ ]:
# Вычисляем корреляционную матрицу
numeric_cols = data.select_dtypes(include=[np.number]).columns
corr_matrix = data[numeric_cols].corr()

# Визуализация
plt.figure(figsize=(14, 12))

# Маска для верхнего треугольника
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Тепловая карта
sns.heatmap(corr_matrix, 
            mask=mask,
            annot=True, 
            fmt='.2f', 
            cmap='RdBu_r',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={'shrink': 0.8})

plt.title('Корреляционная матрица метрик кода', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../images/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Корреляция с целевой переменной
if target_col in corr_matrix.columns:
    target_corr = corr_matrix[target_col].drop(target_col).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 6))
    colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in target_corr.values]
    
    bars = plt.barh(range(len(target_corr)), target_corr.values, color=colors)
    plt.yticks(range(len(target_corr)), target_corr.index)
    plt.xlabel('Корреляция с целевой переменной')
    plt.title('Влияние метрик на наличие дефектов', fontsize=14, fontweight='bold')
    
    # Добавляем значения
    for i, (bar, val) in enumerate(zip(bars, target_corr.values)):
        plt.text(val + (0.02 if val > 0 else -0.07), i, f'{val:.3f}', 
                va='center', fontweight='bold')
    
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig('../images/target_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n📊 Топ-5 метрик, наиболее связанных с дефектами:")
    for metric, corr in target_corr.head().items():
        print(f"  {metric}: {corr:.4f}")

## 6. Сравнение дефектных и чистых модулей

Сравним распределение метрик для модулей с дефектами и без.

In [ ]:
# Boxplot сравнение
if available_metrics:
    # Выбираем топ-4 метрики по корреляции
    if target_col in corr_matrix.columns:
        top_metrics = target_corr.head(4).index.tolist()
    else:
        top_metrics = available_metrics[:4]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    for i, metric in enumerate(top_metrics):
        ax = axes[i]
        
        # Создаем DataFrame для boxplot
        plot_data = pd.DataFrame({
            'Метрика': data[metric],
            'Класс': data[target_col].map({0: 'Без дефектов', 1: 'С дефектами'})
        })
        
        # Логарифмическая шкала для некоторых метрик
        if metric in ['loc', 'halstead_volume']:
            plot_data['Метрика'] = np.log1p(plot_data['Метрика'])
            ylabel = f'log({metric}+1)'
        else:
            ylabel = metric
        
        # Boxplot
        sns.boxplot(data=plot_data, x='Класс', y='Метрика', ax=ax, palette=['#2ecc71', '#e74c3c'])
        ax.set_title(f'Сравнение {metric}', fontweight='bold')
        ax.set_ylabel(ylabel)
        ax.set_xlabel('')
        ax.grid(True, alpha=0.3)
        
        # Добавляем статистику
        stats = data.groupby(target_col)[metric].describe()
        print(f"\n{metric}:")
        print(stats)
    
    plt.tight_layout()
    plt.savefig('../images/comparison_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Анализ выбросов

Исследуем наличие выбросов в данных.

In [ ]:
# Функция для обнаружения выбросов методом IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print("🔍 АНАЛИЗ ВЫБРОСОВ")
print("=" * 50)

outlier_stats = []
for metric in available_metrics:
    n_outliers, lower, upper = detect_outliers_iqr(data, metric)
    pct_outliers = (n_outliers / len(data)) * 100
    outlier_stats.append({
        'Метрика': metric,
        'Выбросы': n_outliers,
        'Процент': f'{pct_outliers:.2f}%',
        'Нижняя граница': f'{lower:.2f}',
        'Верхняя граница': f'{upper:.2f}'
    })

outlier_df = pd.DataFrame(outlier_stats)
print(outlier_df.to_string(index=False))

## 8. Подготовка данных для машинного обучения

Разделяем данные на обучающую и тестовую выборки.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Подготовка признаков и целевой переменной
feature_cols = [c for c in numeric_cols if c != target_col]
X = data[feature_cols]
y = data[target_col]

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Масштабирование признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("📊 ПОДГОТОВКА ДАННЫХ ДЛЯ ML")
print("=" * 50)
print(f"Обучающая выборка: {X_train.shape}")
print(f"Тестовая выборка: {X_test.shape}")
print(f"\nРаспределение в обучающей выборке:")
print(y_train.value_counts(normalize=True).map('{:.2%}'.format))

## 9. Базовое моделирование

Проверим несколько простых моделей для baseline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Обучаем несколько моделей
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100)
}

results = []

for name, model in models.items():
    # Обучение
    model.fit(X_train_scaled, y_train)
    
    # Предсказания
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Метрики
    roc_auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Модель': name,
        'ROC-AUC': f'{roc_auc:.4f}',
        'Точность': f'{model.score(X_test_scaled, y_test):.4f}'
    })
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Defect', 'Defect']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['No Defect', 'Defect'],
                yticklabels=['No Defect', 'Defect'])
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

print("\n📊 СРАВНЕНИЕ МОДЕЛЕЙ")
print("=" * 50)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 10. Выводы и рекомендации

### Основные выводы:

In [ ]:
print("📌 ОСНОВНЫЕ ВЫВОДЫ ПО АНАЛИЗУ")
print("=" * 50)

# Вывод 1: Баланс классов
minority_ratio = min(class_counts.values) / len(data)
if minority_ratio < 0.2:
    print("\n1️⃣ ДИСБАЛАНС КЛАССОВ")
    print(f"   • Доля дефектных модулей: {minority_ratio:.2%}")
    print("   • Рекомендация: Использовать методы балансировки (SMOTE, class_weight)")

# Вывод 2: Важные признаки
if target_col in corr_matrix.columns:
    important_features = target_corr[abs(target_corr) > 0.1].index.tolist()
    print("\n2️⃣ НАИБОЛЕЕ ВАЖНЫЕ ПРИЗНАКИ")
    print(f"   • Обнаружено {len(important_features)} признаков с |корреляцией| > 0.1")
    for feat in important_features[:5]:
        corr_val = target_corr[feat]
        print(f"     - {feat}: {corr_val:.3f}")

# Вывод 3: Выбросы
total_outliers = sum([int(s.split()[0]) for s in outlier_stats if 'Выбросы' in s])
if total_outliers > 0:
    print("\n3️⃣ ВЫБРОСЫ В ДАННЫХ")
    print(f"   • Обнаружены выбросы в нескольких признаках")
    print("   • Рекомендация: Рассмотреть RobustScaler или winsorization")

# Вывод 4: Качество моделей
best_model = results_df.loc[results_df['ROC-AUC'].astype(float).idxmax()]
print("\n4️⃣ КАЧЕСТВО МОДЕЛЕЙ")
print(f"   • Лучшая модель: {best_model['Модель']}")
print(f"   • ROC-AUC: {best_model['ROC-AUC']}")
print("   • Рекомендация: Для улучшения попробовать Gradient Boosting")

# Финальная рекомендация
print("\n" + "="*50)
print("🚀 СЛЕДУЮЩИЕ ШАГИ")
print("="*50)
print("""
1. Применить методы балансировки классов
2. Выполнить отбор признаков
3. Попробовать более сложные модели (XGBoost, LightGBM)
4. Провести кросс-валидацию
5. Оптимизировать гиперпараметры
""")